In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import keras
from keras import layers
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Reading all the cleaned datasets (minus one).
df_00 = pd.read_csv(
    "../dataset/cleaned/Iisingintie0.csv",
    index_col = 0
)
df_01 = pd.read_csv(
    "../dataset/cleaned/Iisingintie1.csv",
    index_col = 0
)
df_02 = pd.read_csv(
    "../dataset/cleaned/Iisingintie2.csv",
    index_col = 0
)
df_04 = pd.read_csv(
    "../dataset/cleaned/Sinetänsalmentie3.csv",
    index_col = 0
)
df_05 = pd.read_csv(
    "../dataset/cleaned/Sinetänsalmentie0.csv",
    index_col = 0
)
df_06 = pd.read_csv(
    "../dataset/cleaned/Sinetänsalmentie1.csv",
    index_col = 0
)
df_07 = pd.read_csv(
    "../dataset/cleaned/Sinetänsalmentie2.csv",
    index_col = 0
)

# Now we split the datasets themselves instead of temporally splitting.
train_dfs = [df_00, df_02, df_01, df_05, df_06, df_07]
test_dfs = [df_04]

df_01

,Loc(m),CElev(m),Roll(deg),Pitch(deg),Yaw(deg),AccX,AccY,AccZ,GyrX,GyrY,GyrZ,Speed_kmh,Frostbumps
Timer(ms),,,,,,,,,,,,,
45900476.5,3760.479,109.111,2.0162,-1.8280,5.3424,0.30559,0.35245,9.83279,-0.00746,0.00199,0.00196,0.2196,0
45900479.0,3760.479,109.112,2.0173,-1.8276,5.3428,0.28938,0.37702,9.83549,0.00233,0.00402,-0.00203,0.2196,0
45900474.0,3760.479,109.111,2.0165,-1.8282,5.3417,0.31479,0.37302,9.79574,-0.00510,0.00201,0.00173,0.2196,0
45900481.5,3760.479,109.112,2.0172,-1.8274,5.3428,0.32006,0.39893,9.81955,-0.00597,0.00235,-0.00437,0.2196,0
45900471.5,3760.479,109.111,2.0165,-1.8283,5.3408,0.31482,0.38664,9.83676,-0.00245,0.00137,-0.00626,0.2196,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46105337.4,6910.462,119.601,2.5315,-3.0370,158.5733,-1.88428,0.21240,9.64411,-0.01138,-0.00057,-0.00476,27.9108,0
46105339.9,6910.481,119.601,2.5313,-3.0349,158.5734,-2.08760,0.48818,9.84319,-0.00640,0.01543,-0.00437,27.9108,0
46105342.4,6910.500,119.601,2.5311,-3.0331,158.5734,-2.34885,0.42224,10.22482,-0.00644,0.01380,-0.00492,27.9108,0


In [7]:
'''
Returns two pandas 3D panels, split as X y which are sktime library-compliant
data structures used for machine learning models. The time step is determined
by the `sequence length`. 

dfs             = list of pandas dataframes; these are the dataframes to be converted.
sequence_length = int; time-step, determines the temporal step in rows.
target_column   = string; the name of the target column
channels_first  = boolean; default is (samples,features,timesteps), otherwise
                  it is (samples, timesteps, features)
'''
def make_panels(dfs, sequence_length, target_column, stride=None, channels_first=True):
    stride = stride or sequence_length
    X_panels, y_panels = {}, {}

    for i, df in enumerate(dfs):
        X = df.drop(columns=[target_column]).to_numpy(dtype=np.float32)
        y = df[target_column].to_numpy()

        starts = range(0, len(X) - sequence_length + 1, stride)
        X_panel = np.stack([X[s:s + sequence_length] for s in starts])      # (n, time, feat)
        y_panel = np.array([y[s:s + sequence_length].max() for s in starts])  # 1 if any bump

        if channels_first:
            X_panel = X_panel.transpose(0, 2, 1)                             # (n, feat, time)

        X_panels[i], y_panels[i] = X_panel, y_panel
    return X_panels, y_panels


'''
Combines lists of pandas panels to a singular X y concatenated
list of values, necessary before inserting it to any ml
models.

X_panels   = list of pandas sequential panels: Features. 
y_panels   = list of pandas sequential panels: For the target variable. 
'''
def combine_panels(X_panels, y_panels):

    X = np.concatenate(
        list(X_panels.values()),
        axis=0
    )

    y = np.concatenate(
        list(y_panels.values()),
        axis=0
    )

    return X, y

In [8]:
# CHANGE THIS NUMBER TO CHANGE THE TIMESTEP
sequence_length = 40

# Create training panels
X_train_panels, y_train_panels = make_panels(
    train_dfs,
    sequence_length,
    "Frostbumps",
    channels_first=False
)

# Create test panels
X_test_panels, y_test_panels = make_panels(
    test_dfs,
    sequence_length,
    "Frostbumps",
    channels_first=False
)

# Combining the panels
X_train, y_train = combine_panels(X_train_panels, y_train_panels)
X_test, y_test = combine_panels(X_test_panels, y_test_panels)

# Set appropriate data types (Keras needs this, it doesn't like float64)
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)


# Scaling. 
# Flattening the sequences to 2D --> transforming it --> 3D
# NOTE for me: I probably could have flattened it at the beginning. 
scaler = StandardScaler()
n_features = X_train.shape[2]
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

In [9]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, classification_report
from tensorflow import keras
from tensorflow.keras import layers

TARGET = "Frostbumps"

# train_dfs / val_dfs / test_dfs: lists of feature DataFrames from prep(),
# each with the TARGET column added back, and no Timer(ms) or Loc(m) columns
feat_cols = [c for c in train_dfs[0].columns if c != TARGET]

# Scale features using TRAIN files only
scaler = StandardScaler().fit(pd.concat(train_dfs)[feat_cols])

def scaled(dfs):
    out = []
    for d in dfs:
        d = d.copy()
        d[feat_cols] = scaler.transform(d[feat_cols])
        out.append(d)
    return out

train_s, val_s, test_s = scaled(train_dfs), scaled(val_dfs), scaled(test_dfs)

WIN = 128
Xtr_p, ytr_p = make_panels(train_s, WIN, TARGET, stride=16, channels_first=False)
Xva_p, yva_p = make_panels(val_s,   WIN, TARGET, channels_first=False)
Xte_p, yte_p = make_panels(test_s,  WIN, TARGET, channels_first=False)

Xtr, ytr = combine_panels(Xtr_p, ytr_p)
Xva, yva = combine_panels(Xva_p, yva_p)
Xte, yte = combine_panels(Xte_p, yte_p)

ytr, yva, yte = [a.astype("float32") for a in (ytr, yva, yte)]
print(Xtr.shape, "bump-window fraction:", ytr.mean(), yva.mean(), yte.mean())

# Small bidirectional GRU
model = keras.Sequential([
    layers.Input(shape=Xtr.shape[1:]),                 # (WIN, n_features)
    layers.Conv1D(16, 5, padding="same", activation="relu"),   # optional local feature extractor
    layers.Bidirectional(layers.GRU(32)),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"),
             keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall")],
)

# Handle class imbalance
pos = ytr.mean()
class_weight = {0: 1.0, 1: (1 - pos) / pos}

model.fit(
    Xtr, ytr,
    validation_data=(Xva, yva),
    epochs=60, batch_size=64,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping("val_pr_auc", mode="max",
                                             patience=8, restore_best_weights=True)],
)

# Evaluate
pva = model.predict(Xva).ravel()
pte = model.predict(Xte).ravel()
print("val  PR-AUC:", average_precision_score(yva, pva))
print("test PR-AUC:", average_precision_score(yte, pte))

# Choose the threshold on VALIDATION, then apply it to test
thresholds = np.linspace(0.05, 0.95, 19)
f1s = [(2 * ((pva > t) & (yva == 1)).sum()) /
       max(1, (pva > t).sum() + (yva == 1).sum()) for t in thresholds]
best_t = thresholds[int(np.argmax(f1s))]
print("threshold:", best_t)
print(classification_report(yte, (pte > best_t).astype(int)))

NameError: name 'val_dfs' is not defined

In [ ]:
TARGET = "Frostbumps"

def load_dfs(file_list):
    dfs = []
    for path in file_list:
        feats, y = prep(path)        # prep returns (features_df, labels_array)
        feats = feats.copy()
        feats[TARGET] = y            # add the label column back
        dfs.append(feats)
    return dfs

train_dfs = load_dfs(train_files)
test_dfs  = load_dfs(test_files)

# Quick sanity check
for name, split in [("train", train_dfs), ("val", val_dfs), ("test", test_dfs)]:
    print(name, [len(d) for d in split], "bump samples:", [int(d[TARGET].sum()) for d in split])
print(train_dfs[0].columns.tolist())

NameError: name 'train_files' is not defined

In [11]:
train_dfs_all = load_dfs(train_dfs)   # all of your friend's training files
test_dfs      = load_dfs(test_dfs)

train_dfs = train_dfs_all[:-1]          # the rest stay as training

NameError: name 'prep' is not defined